## Get Model PTQ Quantization

# GET model Quantization

In [ ]:
%pip install -q omegaconf einops scikit-learn medpy pytorch-lightning "albumentations==1.0.3"
%pip install -q nvidia-modelopt onnx "tensorrt<11" huggingface_hub requests

In [ ]:
import os, sys, copy
import torch
import torch.nn.functional as F

# Colab: mount Drive and use the Drive copy of the repo. Local: use the Mac path.
if os.path.exists("/content"):
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/CSCE-491"
else:
    ROOT = "/Users/bao/Desktop/Y3/CSCE-491"
sys.path.insert(0, os.path.join(ROOT, "GET-Quantization"))   # so `import ptq_utils` works from anywhere
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from ptq_utils import (SKIP, is_target, get_layer_group, load_models, get_dataloaders,
                       forward_pipeline, evaluate, estimate_int8_size_mb, get_state_dict_size_mb)

ModuleNotFoundError: No module named 'ptq_utils'

In [12]:
# Tensor quantization
import torch

x = torch.randn(10)
s = (x.max() - x.min()) / 255 # scale
zp = torch.round(-x.min() / s)  # representation of floating 0.0 in the quantized scale
x_q = torch.clamp(torch.round(x / s) + zp, 0, 255) 
x_hat = (x_q - zp) * s # dequantize

print("Tensor:", x)
print("Scale:", s)
print("Zero:", zp)
print("Quantized x:", x_q)
print("Dequantized x:", x_hat)
print("Error", (x - x_hat).abs().max(), s / 2) # Error <= s / 2 is tolerable



# Shape: (out_channels, in_channels, height, width)
w = torch.randn(16, 3, 5, 5)

# These two lines do the exact same thing:
scale_dim  = w.abs().amax(dim=[1, 2, 3]) / 127
scale_axis = w.abs().amax(axis=[1, 2, 3]) / 127

# This is also the same as it flattens the vector from dimension 1 forward and find the max of the newly collapsed column
# example 16 x 3 x 3 x 3 = 16 x 9, and find max of dimension 9 (columns)
scale_dim = w.abs().flatten(1).amax(dim=1) / 127 

print(scale_dim)
print(scale_axis)
# Both result in a flat tensor of 16 channel scales
print(torch.equal(scale_dim, scale_axis))  # True



Tensor: tensor([-1.2428, -0.3277, -0.6021,  0.0684,  1.5674,  2.1196, -1.1530, -0.6455,
        -0.9537,  1.0434])
Scale: tensor(0.0132)
Zero: tensor(94.)
Quantized x: tensor([  0.,  69.,  48.,  99., 213., 255.,   7.,  45.,  22., 173.])
Dequantized x: tensor([-1.2395, -0.3296, -0.6065,  0.0659,  1.5691,  2.1229, -1.1472, -0.6461,
        -0.9494,  1.0417])
Error tensor(0.0058) tensor(0.0066)
tensor([0.0165, 0.0199, 0.0250, 0.0197, 0.0209, 0.0208, 0.0191, 0.0192, 0.0206,
        0.0180, 0.0177, 0.0171, 0.0308, 0.0166, 0.0183, 0.0200])
tensor([0.0165, 0.0199, 0.0250, 0.0197, 0.0209, 0.0208, 0.0191, 0.0192, 0.0206,
        0.0180, 0.0177, 0.0171, 0.0308, 0.0166, 0.0183, 0.0200])
True


In [ ]:
# Per output channel scale
# Symmetric vs Asymmetric quantization
# Symmetric keeps the center around 0.0
# - use amax()
# Asymmetric maps the center from the 0.0 to a number inside the floating range
# - use standard affine transform formula


# s = w.abs().amax(dim=(1,2,3), keep_dim=True) / 127

# Functional 2D Convolution
# F.conv2d(input, weight=filters, bias, stride, padding, dilation, groups)

# Normal 2D Convolution
# nn.conv2d(in_channel, out_channel, kernel_size, ...)

def fake_quantize_weight(w, per_channel):
    """ 
    Symmetric quantization
    Conv2d weight = [Cout, Cin, kH, kW]
    Return dequantized weight
    """
    if per_channel:
        # Find symmetric scale
        # keepdim=True -> s has shape [Cout,1,1,1] so it broadcasts against w [Cout,Cin,kH,kW]
        s = w.abs().amax(dim=(1,2,3), keepdim=True).clamp(min=1e-8) / 127
    else:
        # per-tensor: one scalar scale for the whole weight (still symmetric, zero_point = 0)
        s = w.abs().amax().clamp(min=1e-8) / 127

    # symmetric: no zero point
    w_q = torch.clamp(torch.round(w / s), -127, 127)
    # Dequantize
    w_hat = w_q * s

    # every element's error must be within half a quantization step
    assert ((w - w_hat).abs() <= s / 2 + 1e-6).all()
    return w_hat

class QuantConv2d(torch.nn.Module):
    """
    Drop-in replacement for nn.Conv2d that simulates INT8 ("fake quant").
      quant_act=False -> Stage A: weight-only (W8A32)
      quant_act=True  -> Stage B: weights AND the conv's input activation (W8A8)
    Math stays in float; we only round values onto the INT8 grid and map them back.
    """
    def __init__(self, conv, per_channel=True, quant_act=False, act_method="minmax") -> None:
        super().__init__() # Call PyTorch's instantiation method
        # buffer = saved with state_dict and moved by .to(device), but not trained
        self.register_buffer("w_hat", fake_quantize_weight(conv.weight.detach(), per_channel))
        bias = conv.bias.detach().clone() if conv.bias is not None else None
        self.register_buffer("bias", bias)   # None is allowed (define `bias` BEFORE registering it)
        # running activation range filled in during calibration
        self.register_buffer("x_min", torch.tensor(float("inf")))
        self.register_buffer("x_max", torch.tensor(float("-inf")))
        # frozen activation quantization params computed by finalize()
        self.register_buffer("scale_a", torch.tensor(1.0))
        self.register_buffer("zp_a", torch.tensor(0.0))
        # these are tuples like (1,1), never None
        self.stride = conv.stride
        self.padding = conv.padding
        self.dilation = conv.dilation
        self.groups = conv.groups
        self.quant_act = quant_act # quant_act = True : apply quantization to activation layer as well
        self.act_method = act_method   # "minmax" | "percentile" | "mse"  (how finalize() picks the range)
        self.calibrating = False
        self._samples = []             # small activation samples, only kept for percentile / mse

    @torch.no_grad()
    def finalize(self):
        """
        Turn the observed activation range into a frozen (scale_a, zp_a). No arguments.
        Call once after the calibration images have been pushed through the model.
          scale = (hi - lo) / 255          zp = round(-lo / scale)  (the integer that means real 0.0)
        """
        lo, hi = self.x_min.item(), self.x_max.item()
        if lo == float("inf"):             # layer never ran during calibration: keep identity-ish defaults
            self.calibrating = False
            return

        if self.act_method != "minmax" and self._samples:
            s = torch.cat(self._samples).float().to(self.x_min.device)
            if self.act_method == "percentile":
                # clip outliers: ignore the most extreme 0.01% on each side
                lo = torch.quantile(s, 1e-4).item()
                hi = torch.quantile(s, 1 - 1e-4).item()
            elif self.act_method == "mse":
                # try shrinking the range; keep the clip that minimises quantization MSE on the samples
                best = float("inf")
                lo0, hi0 = min(lo, 0.0), max(hi, 0.0)
                for alpha in torch.linspace(0.5, 1.0, 26).tolist():
                    c_lo, c_hi = lo0 * alpha, hi0 * alpha
                    sc = max(c_hi - c_lo, 1e-8) / 255
                    zp = min(max(round(-c_lo / sc), 0), 255)
                    s_hat = (torch.clamp(torch.round(s / sc) + zp, 0, 255) - zp) * sc
                    err = torch.mean((s - s_hat) ** 2).item()
                    if err < best:
                        best, lo, hi = err, c_lo, c_hi
        self._samples = []                 # free memory

        lo, hi = min(lo, 0.0), max(hi, 0.0)   # make sure 0.0 is inside the range (so zp stays in [0,255])
        scale = max(hi - lo, 1e-8) / 255
        zp = min(max(round(-lo / scale), 0), 255)
        self.scale_a.fill_(scale)
        self.zp_a.fill_(zp)
        self.calibrating = False

    def forward(self, x):
        if self.calibrating:
            # observe only: track min/max across batches, run the layer in FP32 so later layers see real data
            self.x_min = torch.minimum(self.x_min, x.amin())
            self.x_max = torch.maximum(self.x_max, x.amax())
            if self.act_method != "minmax":
                flat = x.detach().flatten()
                idx = torch.randint(0, flat.numel(), (min(4096, flat.numel()),), device=flat.device)
                self._samples.append(flat[idx].cpu())
        # W8A8 (both weight and activation): quantize -> dequantize the input
        elif self.quant_act:
            q = torch.clamp(torch.round(x / self.scale_a) + self.zp_a, 0, 255)
            x = (q - self.zp_a) * self.scale_a
        # order: input, weight, bias, stride, padding, dilation, groups
        return F.conv2d(x, self.w_hat, self.bias, self.stride, self.padding, self.dilation, self.groups)


@torch.no_grad()
def calibrate(vae, mapper, calib_loader, scale, device="cpu"):
    """
    Stage B calibration. Needs the WHOLE pipeline (the mapper's input comes from the VAE encoder),
    so it lives outside QuantConv2d. Order: observe -> run images -> finalize().
    """
    # .modules() yields modules only (named_modules() would yield (name, module) tuples)
    qlayers = [m for model in (vae, mapper)
               for m in model.modules() if isinstance(m, QuantConv2d)]

    for m in qlayers:           # Switch to observe mode
        m.calibrating = True

    for batch in calib_loader:  # Push real images through the full pipeline
        img = (2 * batch["img"] - 1).to(device)
        forward_pipeline(vae, mapper, img, scale)

    for m in qlayers:           # Freeze scale/zp and switch back to quantize mode
        m.finalize()

def _set_submodule(root, dotted_name, new_module):
    """Replace root.<a>.<b>.<c> given 'a.b.c'."""
    *parent_path, last = dotted_name.split(".")
    parent = root
    for p in parent_path:
        parent = getattr(parent, p)
    setattr(parent, last, new_module)


def swap_model(model, per_channel=True, quant_act=False, act_method="minmax",
               skip_set=SKIP, filter_fn=None, verbose=False):
    """
    Returns a deep copy of `model` where every target Conv2d is replaced by QuantConv2d.
      skip_set  : layer names that must stay FP32 (default SKIP; extend it after the sensitivity sweep)
      filter_fn : optional fn(name)->bool to quantize only some layers (used by the sweep)
    """
    model_q = copy.deepcopy(model)   # keep the FP32 baseline untouched
    # collect first, swap after: don't modify the module tree while iterating it
    targets = [(n, m) for n, m in model_q.named_modules()
               if is_target(n, m, skip_set) and (filter_fn is None or filter_fn(n))]
    # Replace every layer in the target with the fake-quantized node
    for name, m in targets:
        if verbose:
            print(name, m)
        _set_submodule(model_q, name, QuantConv2d(m, per_channel, quant_act, act_method))
    return model_q


# Self-test: QuantConv2d should closely match nn.Conv2d
conv = torch.nn.Conv2d(3, 8, 3, padding=1)
qconv = QuantConv2d(conv, per_channel=True)
xt = torch.randn(1, 3, 16, 16)
print("max abs diff:", (conv(xt) - qconv(xt)).abs().max())

# Self-test (Stage B): observe -> finalize -> quantized forward should still be close
qconv_a = QuantConv2d(conv, per_channel=True, quant_act=True)
qconv_a.calibrating = True
qconv_a(xt)                       # observe pass (runs plain conv)
qconv_a.finalize()                # freezes scale_a / zp_a and leaves calibrating=False
print("scale_a:", qconv_a.scale_a.item(), "zp_a:", qconv_a.zp_a.item())
print("W8A8 max abs diff:", (conv(xt) - qconv_a(xt)).abs().max())



NameError: name 'F' is not defined

In [ ]:
# ============ Stage A: weight-only quantization, no calibration (W8A32) ============
vae, mapper, scale, cfg = load_models(ROOT, config_rel_path="GET/configs/busi_valid.yaml", device=DEVICE)
calib_loader, test_loader = get_dataloaders(cfg["resolved_pickle_path"], n_calib=64, n_eval=50)

results = []   # one row per experiment -> final table in the Report cell
def log_result(name, metrics, size_mb=None):
    results.append({"config": name, "device": DEVICE, **metrics, "MB": size_mb})
    print(f"{name:45s}", {k: round(v, 4) for k, v in metrics.items()})

# Sizes: estimate on the ORIGINAL FP32 models. QuantConv2d is not nn.Conv2d, so the estimator
# would not see the swapped layers, and its fake-quant weights are still stored as FP32 anyway.
vae_fp32_mb, vae_int8_mb = get_state_dict_size_mb(vae), estimate_int8_size_mb(vae)
map_fp32_mb, map_int8_mb = get_state_dict_size_mb(mapper), estimate_int8_size_mb(mapper)
print(f"VAE    FP32 {vae_fp32_mb:.1f} MB -> INT8 (estimated) {vae_int8_mb:.1f} MB")
print(f"Mapper FP32 {map_fp32_mb:.2f} MB -> INT8 (estimated) {map_int8_mb:.2f} MB")

# Baseline
log_result("FP32 baseline", evaluate(vae, mapper, test_loader, scale, DEVICE), vae_fp32_mb + map_fp32_mb)

# Per-tensor vs per-channel; mapper only, VAE only, then both (to see which one costs accuracy)
for per_channel in (False, True):
    tag = "per-channel" if per_channel else "per-tensor"
    mapper_q = swap_model(mapper, per_channel=per_channel)
    vae_q = swap_model(vae, per_channel=per_channel)
    log_result(f"A {tag}: mapper only", evaluate(vae, mapper_q, test_loader, scale, DEVICE))
    log_result(f"A {tag}: VAE only", evaluate(vae_q, mapper, test_loader, scale, DEVICE))
    log_result(f"A {tag}: VAE + mapper", evaluate(vae_q, mapper_q, test_loader, scale, DEVICE),
               vae_int8_mb + map_int8_mb)

NameError: name 'torch' is not defined

In [ ]:
# ============ Stage B: simulated W8A8 (weights + activations, calibrated) ============
# B1. Compare activation-range estimators on the full pipeline
for act_method in ("minmax", "percentile", "mse"):
    vae_q = swap_model(vae, per_channel=True, quant_act=True, act_method=act_method)
    mapper_q = swap_model(mapper, per_channel=True, quant_act=True, act_method=act_method)
    calibrate(vae_q, mapper_q, calib_loader, scale, DEVICE)   # observe -> finalize
    log_result(f"B W8A8 {act_method}", evaluate(vae_q, mapper_q, test_loader, scale, DEVICE),
               vae_int8_mb + map_int8_mb)

# Pick the estimator with the best DSC for the sweep below
best_method = max((r for r in results if r["config"].startswith("B W8A8")), key=lambda r: r["DSC"])["config"].split()[-1]
print("best activation estimator:", best_method)

# B2. Sensitivity sweep: quantize ONE layer group at a time (everything else FP32), see which group hurts DSC
def groups_in(model, tag):
    return sorted({get_layer_group(tag, n) for n, m in model.named_modules() if is_target(n, m)})

base_dsc = results[0]["DSC"]
sweep = []
for tag, model in (("vae", vae), ("mapper", mapper)):
    for g in groups_in(model, tag):
        only_g = lambda n, g=g, tag=tag: get_layer_group(tag, n) == g   # g=g freezes the loop variable
        qm = swap_model(model, per_channel=True, quant_act=True, act_method=best_method, filter_fn=only_g)
        v, mp = (qm, mapper) if tag == "vae" else (vae, qm)
        calibrate(v, mp, calib_loader, scale, DEVICE)
        m = evaluate(v, mp, test_loader, scale, DEVICE)
        sweep.append((g, base_dsc - m["DSC"], m))
        print(f"{g:20s} DSC drop {base_dsc - m['DSC']:+.4f}")

print("\nWorst groups first:")
for g, drop, _ in sorted(sweep, key=lambda t: -t[1]):
    print(f"  {g:20s} {drop:+.4f}")

In [ ]:
# B3. Decide the final skip set from the sweep, re-run W8A8 with it, and save the result
DSC_TOL = 0.01   # a group that costs more than 1 DSC point on its own goes back to FP32
bad_groups = {g for g, drop, _ in sweep if drop > DSC_TOL}
print("groups kept in FP32:", bad_groups or "none")

def names_in_groups(model, tag, groups):
    return {n for n, m in model.named_modules() if is_target(n, m) and get_layer_group(tag, n) in groups}

final_skip = set(SKIP) | names_in_groups(vae, "vae", bad_groups) | names_in_groups(mapper, "mapper", bad_groups)

vae_q = swap_model(vae, True, True, best_method, skip_set=final_skip)
mapper_q = swap_model(mapper, True, True, best_method, skip_set=final_skip)
calibrate(vae_q, mapper_q, calib_loader, scale, DEVICE)
log_result(f"B W8A8 {best_method} + sweep-pruned skip", evaluate(vae_q, mapper_q, test_loader, scale, DEVICE))

# Save skip list + per-layer activation ranges (for the report / reuse after a Colab restart)
ranges = {f"{tag}.{n}": {"x_min": m.x_min.item(), "x_max": m.x_max.item(),
                         "scale_a": m.scale_a.item(), "zp_a": m.zp_a.item()}
          for tag, model in (("vae", vae_q), ("mapper", mapper_q))
          for n, m in model.named_modules() if isinstance(m, QuantConv2d)}
torch.save({"skip": sorted(final_skip), "act_method": best_method, "ranges": ranges},
           os.path.join(ROOT, "GET-Quantization", "calib.pt"))
print("saved", len(ranges), "layer ranges")

In [ ]:
# ============ Stage C: real INT8 on the T4 with TensorRT (Colab GPU only) ============
# torch.ao INT8 is CPU-only; TensorRT is what uses the T4's INT8 Tensor Cores.
# Engines are built for this GPU and are not portable.
import trt_utils as T
assert DEVICE == "cuda", "TensorRT needs the NVIDIA GPU (Colab: Runtime > Change runtime type > T4 GPU)"

OUT = os.path.join(ROOT, "GET-Quantization", "trt_out")
os.makedirs(OUT, exist_ok=True)
skip_for_trt = final_skip if "final_skip" in globals() else SKIP   # layers kept FP32 (from the Stage B sweep)

# Split GET into 3 tensor->tensor sub-networks and record the inputs each one sees on real images
wraps = T.make_wrappers(vae, mapper)                                  # {'enc','map','dec'}
calib_inputs = T.collect_stage_inputs(vae, mapper, calib_loader, scale, DEVICE)

for k, wrap in wraps.items():
    example = calib_inputs[k][0].to(DEVICE)                           # fixed shape, batch 1

    # (1) FP baseline: ONNX of the unquantized module -> FP16 TensorRT engine
    T.export_onnx(wrap, example, f"{OUT}/{k}_fp.onnx")
    T.build_engine(f"{OUT}/{k}_fp.onnx", f"{OUT}/{k}_fp16.plan", int8=False, fp16=True)

    # (2) INT8: ModelOpt inserts Q/DQ nodes (calibrated on calib_inputs), export, build INT8 engine.
    #     deepcopy so the FP32 `vae`/`mapper` used above stay untouched.
    qwrap = T.quantize_int8(copy.deepcopy(wrap), calib_inputs[k], DEVICE,
                            skip_set=skip_for_trt, prefix=getattr(wrap, "PREFIX", ""))
    T.export_onnx(qwrap, example, f"{OUT}/{k}_int8.onnx")
    T.build_engine(f"{OUT}/{k}_int8.onnx", f"{OUT}/{k}_int8.plan", int8=True, fp16=True)
    print(k, "engines built")

In [ ]:
# Evaluate the TensorRT pipelines with the SAME evaluate() / test images as every other row.
# forward_fn replaces the PyTorch pipeline; latency uses cuda.synchronize inside evaluate().
def plan_mb(variant):   # total engine size on disk, decimal MB
    return sum(os.path.getsize(f"{OUT}/{k}_{variant}.plan") for k in wraps) / 1e6

for variant in ("fp16", "int8"):
    engines = {k: T.TRTEngine(f"{OUT}/{k}_{variant}.plan") for k in wraps}
    fwd = T.make_trt_forward(engines["enc"], engines["map"], engines["dec"], scale)
    log_result(f"C TensorRT {variant}",
               evaluate(None, None, test_loader, scale, DEVICE, forward_fn=fwd), plan_mb(variant))


In [ ]:
# ============ Report: one table for every experiment ============
import pandas as pd
df = pd.DataFrame(results)
df.insert(3, "dDSC_vs_FP32", df["DSC"] - df.loc[0, "DSC"])   # row 0 is the FP32 baseline
print(df.round(4).to_string(index=False))
df.to_csv(os.path.join(ROOT, "GET-Quantization", "results.csv"), index=False)

# ms/img is only comparable within the same device.